# Qwen3 Coder and Vision on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

The specialist members of the Qwen3 family: three coding models and a
vision-language model. `01-qwen3-core-and-tools.ipynb` covers the general-purpose
models and the shared API mechanics.

**Models covered in this notebook**

| Model ID | Notes |
|---|---|
| `qwen.qwen3-coder-480b-a35b-instruct` | Largest coder, mixture-of-experts (MoE): 480B total / 35B active |
| `qwen.qwen3-coder-next` | Newest coder generation |
| `qwen.qwen3-coder-30b-a3b-instruct` | Small coder, MoE 30B total / 3B active |
| `qwen.qwen3-vl-235b-a22b-instruct` | Vision-language, MoE 235B total / 22B active |

## Which API? Chat Completions.
Bare `/v1` path. The Responses API returns 400 for this family — see
`01-qwen3-core-and-tools.ipynb` §2 for the proof.

## Self-contained, but see also
- **Qwen3 core mechanics, tool use, structured output** →
  `01-qwen3-core-and-tools.ipynb`
- **Auth, the three URL paths, model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, retention/ZDR, CloudWatch** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retries, service tiers** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `repair_tool_arguments` | re-serialises malformed tool-call arguments so they are safe to echo back |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `check_spec` | scores generated code against an expected signature, statically |
| `extract_code_block` | the first fenced code block from a model response |
| `inspect_code` | statically analyses generated Python with `ast`. **Never executes it** |
| `converse_text` | concatenates the text blocks of a Converse response — safer than `content[0]` |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |
| `converse_reasoning` | the reasoning trace from a Converse response, or `""` |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import (
    SLIDE_CALLOUTS,
    SLIDE_KEYWORDS,
    SLIDE_TITLE,
    converse,
    err,
    keyword_recall,
    list_models,
    parse_json_lenient,
    post,
    repair_tool_arguments,
    safe_print,
    slide_data_url,
)

REGION = "us-east-1"

CODER_480B = "qwen.qwen3-coder-480b-a35b-instruct"
CODER_NEXT = "qwen.qwen3-coder-next"
CODER_30B = "qwen.qwen3-coder-30b-a3b-instruct"
VL_235B = "qwen.qwen3-vl-235b-a22b-instruct"

CODERS = [CODER_30B, CODER_NEXT, CODER_480B]

PREFIX = "/v1"  # bare /v1 — not /openai/v1
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)
print("coders  :", CODERS)
print("vision  :", VL_235B)

base URL: https://bedrock-mantle.us-east-1.api.aws/v1
coders  : ['qwen.qwen3-coder-30b-a3b-instruct', 'qwen.qwen3-coder-next', 'qwen.qwen3-coder-480b-a35b-instruct']
vision  : qwen.qwen3-vl-235b-a22b-instruct


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "qwen.qwen3-32b",
    "qwen.qwen3-coder-30b-a3b-instruct",
    "qwen.qwen3-coder-480b-a35b-instruct",
    "qwen.qwen3-coder-next",
    "qwen.qwen3-vl-235b-a22b-instruct",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


qwen.qwen3-32b                         qwen.qwen3-32b-v1:0                      mantle, runtime


qwen.qwen3-coder-30b-a3b-instruct      qwen.qwen3-coder-30b-a3b-v1:0            mantle, runtime


qwen.qwen3-coder-480b-a35b-instruct    -- not on runtime --                     mantle


qwen.qwen3-coder-next                  qwen.qwen3-coder-next                    mantle, runtime


qwen.qwen3-vl-235b-a22b-instruct       qwen.qwen3-vl-235b-a22b                  mantle, runtime



=> 4/5 of these are on bedrock-runtime; 3 under a different id.
   bedrock-mantle only: ['qwen.qwen3-coder-480b-a35b-instruct']
   For those, this notebook's endpoint is the only one that serves them.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


In [3]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Fresh token — expires in <=12h and cannot be refreshed.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

## 1. Code generation

Start with the obvious: write a function to a spec. Note `temperature` low —
code generation benefits from less sampling noise, and this family accepts both
`temperature` and `top_p`.

In [4]:
SPEC = """Write a Python function `chunk_by_tokens(text, max_tokens, overlap=0)` that:
- splits text into chunks of at most max_tokens whitespace-separated tokens
- supports an overlap of N tokens between consecutive chunks
- raises ValueError if overlap >= max_tokens
- returns a list of strings
Include a short docstring. No explanation outside the code block."""

completion = client.chat.completions.create(
    model=CODER_480B,
    messages=[{"role": "user", "content": SPEC}],
    max_tokens=900,
    temperature=0.2,
)
generated = completion.choices[0].message.content or ""
print(generated[:1400])
print("\nusage:", completion.usage.model_dump_json())

```python
def chunk_by_tokens(text, max_tokens, overlap=0):
    """
    Split text into chunks of at most max_tokens whitespace-separated tokens.
    
    Args:
        text (str): The input text to chunk
        max_tokens (int): Maximum number of tokens per chunk
        overlap (int): Number of overlapping tokens between consecutive chunks
        
    Returns:
        list[str]: List of text chunks
        
    Raises:
        ValueError: If overlap >= max_tokens
    """
    if overlap >= max_tokens:
        raise ValueError("overlap must be less than max_tokens")
    
    if not text.strip():
        return []
    
    tokens = text.split()
    if len(tokens) <= max_tokens:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunks.append(' '.join(chunk_tokens))
        
        if end == len(tokens):
            break
            
        st

## 1b. Verify what the model wrote — without running it

A notebook about coding models should check the code, not just print it. The
tempting way is `exec()`. **Do not do that.**

Model output is untrusted input ([OWASP LLM05 — Improper Output
Handling](https://genai.owasp.org/llmrisk/llm052025-improper-output-handling/)),
and this kernel holds your live AWS credentials. `exec()` here is arbitrary code
execution against your own account, triggered by text you did not write. It is
also unnecessary: a specification like the one above is entirely checkable
**statically**.

`inspect_code()` and `check_spec()` in `../_shared/bedrock.py` parse the source
with `ast` and report what it declares. Nothing is executed.

**To genuinely run generated code** you need real isolation — a container or
microVM with no credentials, no network egress, and a CPU/memory cap. AWS Lambda
in a dedicated account, or Bedrock AgentCore's code-interpreter tool, give you
that. A notebook kernel does not.

In [5]:
from bedrock import check_spec, extract_code_block, inspect_code

code_text = extract_code_block(generated)
info = inspect_code(code_text)

print("parses            :", info["parses"], info["error"])
print("functions defined :", info["functions"])
print("raises            :", sorted(set(info["raises"])))
print("imports           :", sorted(set(info["imports"])))

verdict = check_spec(
    code_text,
    function="chunk_by_tokens",
    params=["text", "max_tokens", "overlap"],
    raises="ValueError",  # the spec demanded this guard
)
print("\nspec compliance:")
for check in ("parses", "defines", "signature", "guard"):
    print(f"  {check:10} {'PASS' if verdict[check] else 'FAIL'}")
print(f"  {'overall':10} {'PASS' if verdict['ok'] else 'FAIL'} {verdict['reason']}")

parses            : True 
functions defined : {'chunk_by_tokens': ['text', 'max_tokens', 'overlap']}
raises            : ['ValueError']
imports           : []

spec compliance:
  parses     PASS
  defines    PASS
  signature  PASS
  guard      PASS
  overall    PASS 


## 2. Compare the coders on the same spec

The interesting question is not "can it code" but "how small can I go".

In [6]:
SMALL_SPEC = (
    "Write a Python function `is_palindrome(s)` that ignores case, spaces "
    "and punctuation. Return only the code."
)

# Judged statically: does it parse, and does it define the requested signature?
print(f"{'model':44} {'latency':>9} {'out tok':>8} {'parses':>7} {'signature':>10}")
print("-" * 92)
for model in CODERS:
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": SMALL_SPEC}],
            "max_tokens": 500,
            "temperature": 0.2,
        },
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:44} HTTP {code}: {err(data)[:30]}")
        continue
    body = data["choices"][0]["message"]["content"] or ""
    verdict = check_spec(
        extract_code_block(body), function="is_palindrome", params=["s"]
    )
    print(
        f"{model:44} {elapsed:>8.2f}s "
        f"{data['usage']['completion_tokens']:>8} "
        f"{str(verdict['parses']):>7} {str(verdict['signature']):>10}"
    )

model                                          latency  out tok  parses  signature
--------------------------------------------------------------------------------------------


qwen.qwen3-coder-30b-a3b-instruct                1.50s       58    True       True


qwen.qwen3-coder-next                            7.84s       51    True       True


qwen.qwen3-coder-480b-a35b-instruct              1.17s       58    True       True


## 3. Code review and repair

A more realistic coding workload than generation: find the bug, then fix it.

In [7]:
BUGGY = '''
def average(values):
    """Return the arithmetic mean of a list of numbers."""
    total = 0
    for v in values:
        total += v
    return total / len(values)


def summarise(rows):
    """Return average score per group."""
    groups = {}
    for row in rows:
        groups.setdefault(row["group"], []).append(row["score"])
    return {g: average(scores) for g, scores in groups.items()}
'''

review = client.chat.completions.create(
    model=CODER_NEXT,
    messages=[
        {
            "role": "user",
            "content": (
                "Review this code. Identify every bug or robustness "
                f"problem as a short bulleted list.\n```python\n{BUGGY}```"
            ),
        }
    ],
    max_tokens=600,
    temperature=0.2,
)
print(review.choices[0].message.content[:900])

- **Division by zero**: `average([])` raises `ZeroDivisionError` when `values` is empty (since `len(values)` is 0).  
- **Non-numeric values**: `average()` fails with `TypeError` if `values` contains non-numeric types (e.g., strings, `None`, objects).  
- **Missing keys in rows**: `summarise()` raises `KeyError` if any `row` lacks `"group"` or `"score"` keys.  
- **Mutable default in `setdefault`**: While `setdefault(row["group"], [])` is *technically* safe here (a new list is created per key), it’s a common anti-pattern—`groups.setdefault(row["group"], [])` reuses the same list object if multiple rows share the same group *in the same call*, but this is actually correct here. *(Note: This is not a bug—correction: the `setdefault` usage is fine. The real risk is if `row["group"]` is mutable and reused, but strings are immutable, so it’s safe.)*  
- **No validation of `scores` in `average


In [8]:
# Now ask for a structured repair plan — easier to act on programmatically.
REPAIR_SCHEMA = {
    "type": "object",
    "properties": {
        "bugs": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "function": {"type": "string"},
                    "problem": {"type": "string"},
                    "severity": {"type": "string", "enum": ["low", "medium", "high"]},
                },
                "required": ["function", "problem", "severity"],
                "additionalProperties": False,
            },
        },
        "fixed_code": {"type": "string"},
    },
    "required": ["bugs", "fixed_code"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": CODER_480B,
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the bugs and return fixed code.\n" f"```python\n{BUGGY}```"
                ),
            }
        ],
        "max_tokens": 1800,
        "temperature": 0.2,
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "repair", "strict": True, "schema": REPAIR_SCHEMA},
        },
    },
    region=REGION,
)
choice = (data.get("choices") or [{}])[0]
content = choice.get("message", {}).get("content")
print("HTTP", code, "| finish_reason:", choice.get("finish_reason"))
if content and content.strip():
    repair = parse_json_lenient(content)
    for bug in repair["bugs"]:
        print(f"  [{bug['severity']:6}] {bug['function']}: {bug['problem'][:80]}")
    print("\n--- fixed code (first 500 chars) ---")
    print(repair["fixed_code"][:500])
else:
    print("empty content — raise max_tokens (reasoning consumed the budget)")

HTTP 200 | finish_reason: stop
  [high  ] average: Division by zero error when passed an empty list

--- fixed code (first 500 chars) ---
def average(values):
    """Return the arithmetic mean of a list of numbers."""
    if not values:
        return 0
    total = 0
    for v in values:
        total += v
    return total / len(values)

def summarise(rows):
    """Return average score per group."""
    groups = {}
    for row in rows:
        groups.setdefault(row["group"], []).append(row["score"])
    return {g: average(scores) for g, scores in groups.items()}


## 4. ⚠️ Truncated tool-call arguments — a real, reproducible failure

Before building an agent loop, know about this one. `qwen3-coder` can return
**tool-call arguments that are cut off mid-object** — e.g. `{"path": "calc.py"`
with no closing brace. It is not intermittent: in testing it reproduced on 10 of
10 calls for a simple single-argument tool.

Two consequences:
1. `json.loads()` on the arguments raises.
2. Echoing the assistant message back **verbatim** in the next request gets a
   **400** — the API rejects the malformed JSON you just forwarded.

The fix is to repair the arguments before using *or* echoing them.

In [9]:
probe_tool = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a file from the workspace.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
            },
        },
    }
]

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": CODER_480B,
        "messages": [{"role": "user", "content": "Read calc.py using the tool."}],
        "tools": probe_tool,
        "tool_choice": "auto",
        "max_tokens": 600,
        "temperature": 0.2,
    },
    region=REGION,
)
message = data["choices"][0]["message"]
calls = message.get("tool_calls") or []
if calls:
    raw_args = calls[0]["function"]["arguments"]
    print("raw arguments:", repr(raw_args))
    try:
        json.loads(raw_args)
        print("json.loads     : OK")
    except json.JSONDecodeError as exc:
        print(f"json.loads     : FAILS ({exc.msg})")
    print("lenient parse  :", parse_json_lenient(raw_args))
    print("repaired string:", repair_tool_arguments(raw_args))

raw arguments: '{"path": "calc.py"'
json.loads     : FAILS (Expecting ',' delimiter)
lenient parse  : {'path': 'calc.py'}
repaired string: {"path": "calc.py"}


`bedrock.parse_json_lenient()` closes unbalanced braces and recovers the fields
that did arrive; `bedrock.repair_tool_arguments()` re-serialises them into a string
that is safe to echo. Use the repaired string in the message you send back:

In [10]:
if calls:
    repaired_message = {
        "role": "assistant",
        "tool_calls": [
            {
                **c,
                "function": {
                    **c["function"],
                    "arguments": repair_tool_arguments(c["function"]["arguments"]),
                },
            }
            for c in calls
        ],
    }
    follow_up = [
        {"role": "user", "content": "Read calc.py using the tool."},
        repaired_message,
        {
            "role": "tool",
            "tool_call_id": calls[0]["id"],
            "content": json.dumps({"content": "def divide(a, b):\n    return a / b\n"}),
        },
    ]
    code_ok, _ = post(
        f"{PREFIX}/chat/completions",
        {
            "model": CODER_480B,
            "messages": follow_up,
            "tools": probe_tool,
            "max_tokens": 300,
        },
        region=REGION,
    )
    code_bad, bad = post(
        f"{PREFIX}/chat/completions",
        {
            "model": CODER_480B,
            "messages": [
                follow_up[0],
                {k: v for k, v in message.items() if v is not None},
                follow_up[2],
            ],
            "tools": probe_tool,
            "max_tokens": 300,
        },
        region=REGION,
    )
    print(f"echoing REPAIRED arguments -> HTTP {code_ok}")
    print(f"echoing RAW arguments      -> HTTP {code_bad} {err(bad)[:80]}")

echoing REPAIRED arguments -> HTTP 200
echoing RAW arguments      -> HTTP 400 ErrorEvent { error: APIError { type: "BadRequestError", code: Some(400), message


## 5. An agentic coding loop

Give the model file tools and let it work. This is the shape of a coding agent:
read, write, run, iterate — with the argument repair from §4 built in.

In [11]:
VIRTUAL_FS = {
    "calc.py": (
        "def add(a, b):\n"
        "    return a + b\n\n\n"
        "def divide(a, b):\n"
        "    return a / b\n"
    ),
}


def read_file(path: str) -> dict:
    return {"path": path, "content": VIRTUAL_FS.get(path), "exists": path in VIRTUAL_FS}


def write_file(path: str, content: str) -> dict:
    VIRTUAL_FS[path] = content
    return {"path": path, "bytes": len(content), "ok": True}


def check_file(path: str) -> dict:
    """Statically check that divide() guards against a zero divisor.

    This is the agent's feedback signal, and it is deliberately NOT an exec().
    The file content was written by the model; running it would hand the model
    arbitrary code execution in this kernel (OWASP LLM05). `inspect_code()`
    parses it instead and reports what it declares.

    A real coding agent that must run its own output belongs in a sandbox with
    no credentials and no network - Lambda in an isolated account, or the
    AgentCore code-interpreter tool.
    """
    src = VIRTUAL_FS.get(path)
    if src is None:
        return {"ok": False, "error": "no such file"}
    info = inspect_code(src)
    if not info["parses"]:
        return {"ok": False, "error": f"syntax error: {info['error']}"}
    if "divide" not in info["functions"]:
        return {"ok": False, "error": "divide() not defined"}
    raised = set(info["raises"])
    if not raised:
        return {
            "ok": False,
            "error": "divide() has no raise; a zero divisor would surface a "
            "bare ZeroDivisionError",
        }
    return {"ok": True, "note": f"guards by raising {sorted(raised)}"}

The tool schemas, the dispatch table, and the loop itself. Note the argument
repair from §4 applied *before* echoing the assistant message back.

In [12]:
CODE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a file from the workspace.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write a file to the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "content": {"type": "string"},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_file",
            "description": "Statically check a file against its requirements.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
            },
        },
    },
]

DISPATCH = {"read_file": read_file, "write_file": write_file, "check_file": check_file}

TASK = (
    "The file calc.py fails its checks. Read it, fix divide() so dividing by "
    "zero raises a ValueError with a clear message, write the file back, then "
    "check the file. Use the tools."
)

Two helpers first: one to echo the assistant turn back safely, one to run a
single tool call.

In [13]:
def echo_assistant_turn(calls: list[dict]) -> dict:
    """Rebuild the assistant message with REPAIRED tool arguments.

    Echoing a truncated argument string verbatim is rejected with a 400 (§4), so
    repair before forwarding.
    """
    return {
        "role": "assistant",
        "tool_calls": [
            {
                **c,
                "function": {
                    **c["function"],
                    "arguments": repair_tool_arguments(c["function"]["arguments"]),
                },
            }
            for c in calls
        ],
    }


def run_tool_call(call: dict) -> tuple[str, dict, dict]:
    """Execute one tool call. Returns (name, args, result); never raises."""
    name = call["function"]["name"]
    try:
        args = parse_json_lenient(call["function"]["arguments"] or "{}")
    except ValueError:
        args = {}
    fn = DISPATCH.get(name)
    try:
        result = fn(**args) if fn else {"error": f"unknown tool {name}"}
    except Exception as exc:  # a tool must never kill the loop
        result = {"error": f"{type(exc).__name__}: {exc}"}
    return name, args, result

The loop itself. `max_rounds` is a hard bound — an agent without one can spin
indefinitely on a task it cannot finish (OWASP LLM10, Unbounded Consumption).

In [14]:
def coding_agent(task: str, model: str = CODER_480B, max_rounds: int = 8) -> dict:
    """Read/write/check loop over the virtual workspace, bounded by max_rounds."""
    convo = [{"role": "user", "content": task}]
    trace = []
    for round_no in range(max_rounds):
        code, data = post(
            f"{PREFIX}/chat/completions",
            {
                "model": model,
                "messages": convo,
                "tools": CODE_TOOLS,
                "tool_choice": "auto",
                "max_tokens": 1500,
                "temperature": 0.2,
            },
            region=REGION,
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        message = data["choices"][0]["message"]
        calls = message.get("tool_calls") or []
        if not calls:
            return {
                "answer": message.get("content") or "",
                "trace": trace,
                "rounds": round_no + 1,
            }
        convo.append(echo_assistant_turn(calls))
        for call in calls:
            name, args, result = run_tool_call(call)
            trace.append(
                {
                    "round": round_no + 1,
                    "tool": name,
                    "args": {
                        k: (v[:40] + "…" if isinstance(v, str) and len(v) > 40 else v)
                        for k, v in args.items()
                    },
                    "result": result,
                }
            )
            convo.append(
                {
                    "role": "tool",
                    "tool_call_id": call["id"],
                    "content": json.dumps(result),
                }
            )
    return {"answer": "(max rounds reached)", "trace": trace, "rounds": max_rounds}

Run it, and inspect the trace round by round.

In [15]:
agent_result = coding_agent(TASK)
print(f"rounds: {agent_result['rounds']}")
for step in agent_result["trace"]:
    print(f"  r{step['round']} {step['tool']:11} -> {json.dumps(step['result'])[:90]}")
print("\nfinal file:")
print(VIRTUAL_FS["calc.py"])
print("checks now:", check_file("calc.py"))

rounds: 4
  r1 read_file   -> {"path": "calc.py", "content": "def add(a, b):\n    return a + b\n\n\ndef divide(a, b):\n 
  r2 write_file  -> {"path": "calc.py", "bytes": 133, "ok": true}
  r3 check_file  -> {"ok": true, "note": "guards by raising ['ValueError']"}

final file:
def add(a, b):
    return a + b


def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b
checks now: {'ok': True, 'note': "guards by raising ['ValueError']"}


## 6. Vision — Qwen3-VL

Images go into Chat Completions as `image_url` content blocks. Base64 data URLs
and `s3://` URIs are supported; arbitrary `https://` image URLs are not.

The image is `_shared/assets/aws-summit-slide.jpg`, a frame from a public AWS talk
showing a labelled architecture diagram (provenance in `_shared/bedrock.py`). It
carries real text, so the cells below ask OCR questions whose answers can be
checked, which a solid colour swatch cannot do.

In [16]:
slide = slide_data_url()
print(f"slide data URL: {len(slide)} chars; title is {SLIDE_TITLE!r}\n")

vision = client.chat.completions.create(
    model=VL_235B,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": slide}},
                {
                    "type": "text",
                    "text": "Read this slide. Give its title, then quote the three "
                            "green callout lines.",
                },
            ],
        }
    ],
    max_tokens=260,
)
answer = (vision.choices[0].message.content or "").strip()
hits, total = keyword_recall(answer, SLIDE_CALLOUTS)
print(f"callouts {hits}/{total}, title={SLIDE_TITLE.lower() in answer.lower()}")
print(" ", " ".join(answer.split())[:220])

slide data URL: 43607 chars; title is 'Ingestion from database'



callouts 3/3, title=True
  Title: Ingestion from database Green callout lines: 1. ✓ Pay according to job duration 2. ✓ Lower compute cost, up to 90% 3. ✓ Lower storage cost for rarely accessed data


In [17]:
# Very small images are rejected. Worth knowing so you don't chase a phantom bug.
# A literal 1x1 PNG, so the degenerate input is obvious rather than generated.
TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8"
    "AAxAAADwABbT8HRAAAAABJRU5ErkJggg=="
)
code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": VL_235B,
        "max_tokens": 30,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{TINY_PNG_B64}"},
                    },
                    {"type": "text", "text": "Colour?"},
                ],
            }
        ],
    },
    region=REGION,
)
print(f"1x1 PNG -> HTTP {code} {'ok' if code == 200 else err(data)[:80]}")

1x1 PNG -> HTTP 400 JSON-RPC error -32602: Job registration failed: Unsupported feature: Failed to s


## 7. Structured extraction from an image

The realistic vision workload: turn a picture into typed data. We build a simple
two-band chart so the extraction has something to actually read.

In [18]:
# Structured output over an image. The schema asks for things the slide actually
# contains, so the result is checkable against SLIDE_TITLE and SLIDE_KEYWORDS.
SLIDE_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "aws_services": {"type": "array", "items": {"type": "string"}},
        "flow_direction": {
            "type": "string",
            "enum": ["left_to_right", "right_to_left", "top_to_bottom", "other"],
        },
    },
    "required": ["title", "aws_services", "flow_direction"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": VL_235B,
        "max_tokens": 700,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": slide}},
                    {
                        "type": "text",
                        "text": "Describe this architecture slide: its title, the "
                                "AWS services shown, and the direction the flow "
                                "runs.",
                    },
                ],
            }
        ],
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "slide", "strict": True, "schema": SLIDE_SCHEMA},
        },
    },
    region=REGION,
)
choice = (data.get("choices") or [{}])[0]
content = choice.get("message", {}).get("content")
print("HTTP", code, "| finish_reason:", choice.get("finish_reason"))
if content and content.strip():
    parsed = parse_json_lenient(content)
    print(json.dumps(parsed, indent=2))
    named = ", ".join(parsed.get("aws_services") or [])
    hits, total = keyword_recall(named, SLIDE_KEYWORDS)
    print(f"\nservice recall {hits}/{total}; "
          f"title matches: {SLIDE_TITLE.lower() in (parsed.get('title') or '').lower()}")
else:
    print("empty content — raise max_tokens")

HTTP 200 | finish_reason: stop
{
  "title": "Ingestion from database",
  "aws_services": [
    "AWS Glue (with serverless ETL)",
    "Amazon EMR (with spot instance)",
    "Amazon S3 (Data lake with storage tiers)"
  ],
  "flow_direction": "left_to_right"
}

service recall 4/5; title matches: True


## 8. Which Qwen3 models take images?

Only the VL variant. Sending an image to a text model is not always a clean
error, so check rather than assume.

In [19]:
print(f"{'model':44} {'image accepted':>16}")
print("-" * 62)
for model in [VL_235B, CODER_30B, "qwen.qwen3-32b"]:
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "max_tokens": 30,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": slide}},
                        {"type": "text", "text": "Slide title? Five words."},
                    ],
                }
            ],
        },
        region=REGION,
        attempts=1,
        timeout=60,
    )
    if code == 200:
        answer = (data["choices"][0]["message"]["content"] or "").strip()
        verdict = f"yes ({answer[:12]!r})"
    else:
        verdict = f"{code}" if code != -1 else "stalled"
    print(f"{model:44} {verdict:>16}")

model                                          image accepted
--------------------------------------------------------------


qwen.qwen3-vl-235b-a22b-instruct             yes ('Ingestion fr')


qwen.qwen3-coder-30b-a3b-instruct                         400


qwen.qwen3-32b                                            400


## 9. Regional availability

In [20]:
regions = ("us-east-1", "us-east-2", "us-west-2", "eu-central-1")
inventory = {}
for region in regions:
    try:
        inventory[region] = set(list_models(region))
    except (RuntimeError, OSError) as exc:
        inventory[region] = set()
        print(f"{region}: {type(exc).__name__}")

print(f"{'model':44} " + "  ".join(f"{r:>13}" for r in regions))
print("-" * 104)
for model in CODERS + [VL_235B]:
    cells = "  ".join(
        f"{('yes' if model in inventory[r] else '-'):>13}" for r in regions
    )
    print(f"{model:44} {cells}")

model                                            us-east-1      us-east-2      us-west-2   eu-central-1
--------------------------------------------------------------------------------------------------------
qwen.qwen3-coder-30b-a3b-instruct                      yes            yes            yes            yes
qwen.qwen3-coder-next                                  yes            yes            yes            yes
qwen.qwen3-coder-480b-a35b-instruct                    yes            yes            yes              -
qwen.qwen3-vl-235b-a22b-instruct                       yes            yes            yes            yes


## 10. Production shape

In [21]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "qwen-coder-samples",
        "tags": {"Application": "QwenCoderDemo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)


class QwenCoder:
    """Coding helper: low temperature, schema-validated output, retries."""

    def __init__(self, model=CODER_480B, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def _call(self, messages, max_tokens, schema=None, tools=None):
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0.2,  # code wants low sampling noise
            "service_tier": self.tier,
        }
        if tools:
            body["tools"] = tools
            body["tool_choice"] = "auto"
        if schema:
            body["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "out", "strict": True, "schema": schema},
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429/5xx with exponential backoff.
        code, data = post(
            f"{PREFIX}/chat/completions", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data

    def write_function(
        self,
        spec: str,
        *,
        function: str,
        params: list[str] | None = None,
        max_tokens: int = 900,
    ) -> dict:
        """Generate a function and verify it STATICALLY. Never executes output."""
        data = self._call([{"role": "user", "content": spec}], max_tokens)
        body = data["choices"][0]["message"]["content"] or ""
        source = extract_code_block(body)
        verdict = check_spec(source, function=function, params=params)
        return {"ok": verdict["ok"], "verdict": verdict, "source": source}


coder = QwenCoder(project=project_id)
outcome = coder.write_function(
    "Write a Python function `slugify(s)` that lowercases, replaces non-alphanumerics "
    "with hyphens, and collapses repeats. Return only code.",
    function="slugify",
    params=["s"],
)
print("ok:", outcome["ok"], "|", outcome["verdict"])
print(outcome["source"][:320])

project: 200 proj_fk7yoxtj...


ok: True | {'parses': True, 'defines': True, 'signature': True, 'guard': True, 'ok': True, 'reason': ''}
import re

def slugify(s):
    s = s.lower()
    s = re.sub(r'[^a-z0-9]+', '-', s)
    s = re.sub(r'^-+|-+$', '', s)
    return s


In [22]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived:", code, archived.get("status"))

archived: 200 archived


## Gotchas — Qwen3 Coder and Vision

| Gotcha | Detail |
|---|---|
| Path prefix | Bare `/v1` — Responses API returns 400 for this family |
| Images | Only `qwen3-vl-*`; base64 or `s3://` only, no `https://` URLs |
| Tiny images | A 1×1 PNG is rejected as an unsupported format |
| `content` can be `None` | Check `finish_reason` before slicing/parsing |
| Strict JSON | Parse leniently — models can append text after a valid object |
| Code temperature | Use ~0.2; the family accepts `temperature` and `top_p` |
| **Truncated tool arguments** | `qwen3-coder` truncates mid-object; repair before use AND before echoing |
| Verify generated code | **Statically** (`ast`) — never `exec()` model output in your kernel |
| Tool errors | Return structured errors so the agent loop can recover |
| Region | Check per model; not every coder is in every Region |

## Where next
- `01-qwen3-core-and-tools.ipynb` — the general-purpose Qwen3 models
- Other coding model: `../07-mistral/02-devstral-and-voxtral.ipynb`
- Other vision models: `../10-nvidia-nemotron/`, `../12-writer-palmyra/`

## Converse in earnest — the tool loop, provider parameters, and caching

The earlier endpoint section proved this model answers through Converse. That is
the easy part. This section does the three things you actually need on
`bedrock-runtime`, because each differs from the `bedrock-mantle` equivalent:

1. **A complete tool round trip** — `toolUse` out, `toolResult` back in. Getting a
   tool *call* is half the job; feeding the result back is where the shapes bite.
2. **`additionalModelRequestFields`** — Converse normalises the common fields, so
   anything provider-specific goes through this escape hatch.
3. **`cachePoint`** — prompt caching is a first-class Converse block, and support
   for it is per model rather than universal.


In [23]:
from bedrock import converse_text, converse_tool_uses, resolve_runtime_id, runtime_client

RUNTIME_ID = "qwen.qwen3-coder-30b-a3b-v1"
runtime = runtime_client(REGION)
resolved = resolve_runtime_id(RUNTIME_ID, REGION)

# Converse tool shape: toolSpec, and the JSON Schema nests under inputSchema.json.
# This is NOT the OpenAI shape - there is no {"type": "function"} wrapper.
WEATHER_TOOL = {
    "toolSpec": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }
        },
    }
}

history = [
    {"role": "user", "content": [{"text": "What is the weather in Singapore? Use the tool."}]}
]
first = runtime.converse(
    modelId=resolved,
    messages=history,
    toolConfig={"tools": [WEATHER_TOOL]},
    inferenceConfig={"maxTokens": 500},
)
print("turn 1 stop reason:", first.get("stopReason"))
print("turn 1 blocks     :", [next(iter(b)) for b in first["output"]["message"]["content"]])

uses = converse_tool_uses(first)
if not uses:
    print("no tool call this run - tool_choice defaults to the model's discretion;")
    print("retry, or set toolConfig['toolChoice'] to compel one.")
else:
    use = uses[0]
    print(f"tool call         : {use['name']}({use['input']})")
    # Validate before acting on it. A malformed call still reports tool_use.
    city = str(use["input"].get("city", "")).lower()
    print("arguments valid   :", "yes" if "singapore" in city else f"NO ({use['input']})")

    # Echo the assistant turn back VERBATIM, then answer with a toolResult whose
    # toolUseId matches. Dropping either breaks the loop with a 400.
    history.append(first["output"]["message"])
    history.append(
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": use["toolUseId"],
                        "content": [{"json": {"tempC": 31, "conditions": "humid"}}],
                    }
                }
            ],
        }
    )
    second = runtime.converse(
        modelId=resolved,
        messages=history,
        toolConfig={"tools": [WEATHER_TOOL]},
        inferenceConfig={"maxTokens": 300},
    )
    print("turn 2 stop reason:", second.get("stopReason"))
    print("final answer      :", converse_text(second).strip()[:160])


turn 1 stop reason: tool_use
turn 1 blocks     : ['toolUse']
tool call         : get_weather({'city': 'Singapore'})
arguments valid   : yes


turn 2 stop reason: end_turn
final answer      : The weather in Singapore is currently humid with a temperature of 31°C.


In [24]:
# Converse normalises maxTokens, temperature, topP and stopSequences. Anything
# provider-specific goes through additionalModelRequestFields, unvalidated by
# Converse and passed to the provider as-is. That makes it powerful and sharp:
# a key this model does not recognise is a 400, not a silent no-op.
from bedrock import converse_reasoning

PUZZLE = (
    "A bat and ball cost $1.10 together. The bat costs $1.00 more than the ball. "
    "How much is the ball?"
)

for label, extra in [
    ("no extra fields", None),
    ("provider fields", {"reasoning_effort": "low"}),
]:
    kwargs = {"additionalModelRequestFields": extra} if extra else {}
    try:
        response = runtime.converse(
            modelId=resolved,
            messages=[{"role": "user", "content": [{"text": PUZZLE}]}],
            inferenceConfig={"maxTokens": 900},
            **kwargs,
        )
    except Exception as exc:
        print(f"{label:<16} {type(exc).__name__}: {str(exc)[-90:]}")
        continue
    blocks = [next(iter(b)) for b in response["output"]["message"]["content"]]
    trace = converse_reasoning(response)
    answer = converse_text(response).strip().replace("\n", " ")
    print(f"{label:<16} out={response['usage']['outputTokens']:>4} blocks={blocks}")
    print(f"{'':<16} reasoning={len(trace)} chars | {answer[:70]}")

print()
print("Note whether a reasoningContent block appears above. Some models return the")
print("trace as a typed block on Converse and some do not, so read the blocks")
print("rather than assuming - and never index content[0].")


no extra fields  out= 198 blocks=['text']
                 reasoning=0 chars | Let me work through this step by step.  Let's define: - Ball cost = x 


provider fields  out= 234 blocks=['text']
                 reasoning=0 chars | Let me solve this step by step.  Let's define: - Ball cost = x dollars

Note whether a reasoningContent block appears above. Some models return the
trace as a typed block on Converse and some do not, so read the blocks
rather than assuming - and never index content[0].


In [25]:
# cachePoint is a Converse block, but support for it is per model rather than
# universal. Ask before designing around it.
HANDBOOK = "You are a support handbook. " + (
    "Retries: use exponential backoff with full jitter, cap at 16 seconds. " * 160
)

try:
    usage = runtime.converse(
        modelId=resolved,
        system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
        messages=[{"role": "user", "content": [{"text": "One line: the retry policy?"}]}],
        inferenceConfig={"maxTokens": 60},
    )["usage"]
    print("cachePoint accepted:", {k: v for k, v in usage.items() if "cache" in k.lower()})
except Exception as exc:
    print(f"cachePoint -> {type(exc).__name__}")
    print(f"    {str(exc)[-140:]}")
    print()
    print("Not every model supports prompt caching on Converse. Note the exception")
    print("type: this surfaces as an access or validation error rather than a clear")
    print("'unsupported feature' message, which is easy to misread as a permissions")
    print("problem. Probe it once per model instead of assuming it is available.")

# The same call without the cachePoint block works, so caching is the only part
# that is unavailable.
usage = runtime.converse(
    modelId=resolved,
    system=[{"text": "You are terse."}],
    messages=[{"role": "user", "content": [{"text": "One line: why use backoff?"}]}],
    inferenceConfig={"maxTokens": 60},
)["usage"]
print()
print("same call without cachePoint -> OK, tokens:", usage["totalTokens"])


cachePoint -> AccessDeniedException
    nverse operation: You invoked an unsupported model or your request did not allow prompt caching. See the documentation for more information.

Not every model supports prompt caching on Converse. Note the exception
type: this surfaces as an access or validation error rather than a clear
'unsupported feature' message, which is easy to misread as a permissions
problem. Probe it once per model instead of assuming it is available.



same call without cachePoint -> OK, tokens: 40


### What this section adds over the endpoint check above

- **The tool loop is the part that bites.** `toolSpec` is not the OpenAI shape,
  the JSON Schema nests under `inputSchema.json`, and the second turn must echo the
  assistant message back verbatim alongside a `toolResult` whose `toolUseId`
  matches. Miss any of that and you get a 400.
- **`additionalModelRequestFields` is unvalidated by Converse.** It is the only way
  to reach provider-specific behaviour, and a key the model does not recognise
  fails the call rather than being ignored.
- **Feature support is per model, not per endpoint.** Read the output above rather
  than carrying an assumption over from another family.
